In [11]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

collision_path = Path("..") / "data" / "raw" / "dft-road-casualty-statistics-collision-last-5-years.csv"
casualty_path = Path("..") / "data" / "raw" / "dft-road-casualty-statistics-casualty-last-5-years.csv"
vehicle_path = Path("..") / "data" / "raw" / "dft-road-casualty-statistics-vehicle-last-5-years.csv"

dfcol = pd.read_csv(
    collision_path,
    dtype={"collision_index": "string", "collision_ref_no": "string"},
)
dfcas = pd.read_csv(
    casualty_path,
    dtype={"collision_index": "string", "collision_ref_no": "string"},
)
dfveh = pd.read_csv(
    vehicle_path,
    dtype={
        "collision_index": "string",
        "collision_ref_no": "string",
        "generic_make_model": "string",
    },
)

tables = {"COLLISION": dfcol, "CASUALTY": dfcas, "VEHICLE": dfveh}

# print("============= INFO ==================")
# for name, df in tables.items():
#     print(f"\nТаблица: {name}")
#     print(f"Размер: {df.shape[0]} строк, {df.shape[1]} столбцов")
#     print(f"Память: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

#     print("\nТипы данных:")
#     display(df.dtypes.to_frame(name="Тип данных"))

#     print("Первые 5 строк:")
#     display(df.head(5))
#     print("Последние 5 строк:")
#     display(df.tail(5))

# print("\n============= MISSES ==================")
# for name, df in tables.items():
#     print(f"\nТаблица: {name}")

#     missing = pd.DataFrame({
#         "Количество пропусков": df.isna().sum(),
#         "Процент пропусков": (df.isna().mean() * 100).round(2),
#     }).sort_values(by="Процент пропусков", ascending=False)
#     display(missing)

#     total_missing = int(df.isna().sum().sum())
#     total_cells = int(df.size)
#     cols_with_missing = missing[missing["Количество пропусков"] > 0].index.tolist()
#     print(f"Всего пропусков: {total_missing}")
#     print(f"Общая доля пропусков: {total_missing / total_cells * 100:.4f}%")
#     print(f"Столбцы с пропусками: {cols_with_missing if cols_with_missing else 'нет'}")

# print("\nDUPLICATES")
# dup_rows = []
# for name, df in tables.items():
#     dups = int(df.duplicated().sum())
#     dup_rows.append({
#         "Таблица": name,
#         "Строк": len(df),
#         "Дубликатов": dups,
#         "Есть дубликаты": "Да" if dups > 0 else "Нет",
#     })
# display(pd.DataFrame(dup_rows).set_index("Таблица"))

# print("\nУНИКАЛЬНОСТЬ COLLISION_INDEX")
# display(pd.DataFrame([{
#     "Уникален": bool(dfcol["collision_index"].is_unique),
#     "Уникальных значений": int(dfcol["collision_index"].nunique()),
#     "Всего значений": int(dfcol["collision_index"].size),
# }], index=["COLLISION"]))

# print("\nПОВТОРЕНИЯ COLLISION_INDEX В VEHICLE / CASUALTY")
# rep_rows = []
# for name, df in [("VEHICLE", dfveh), ("CASUALTY", dfcas)]:
#     counts = df["collision_index"].value_counts()
#     rep_rows.append({
#         "Таблица": name,
#         "Уникальных ДТП": int(df["collision_index"].nunique()),
#         "ДТП с >1 записью": int((counts > 1).sum()),
#         "Макс. записей на ДТП": int(counts.max()),
#     })
# display(pd.DataFrame(rep_rows).set_index("Таблица"))

# print("\nСОСТАВНЫЕ ИДЕНТИФИКАТОРЫ")
# composite_keys = {
#     "VEHICLE": (dfveh, ["collision_index", "vehicle_reference"]),
#     "CASUALTY": (dfcas, ["collision_index", "casualty_reference"]),
# }
# ck_rows = []
# for name, (df, keys) in composite_keys.items():
#     if all(col in df.columns for col in keys):
#         dups = int(df.duplicated(subset=keys).sum())
#         ck_rows.append({
#             "Таблица": name,
#             "Ключ": " + ".join(keys),
#             "Дубликатов": dups,
#             "Ключ уникален": "Да" if dups == 0 else "Нет",
#         })
#     else:
#         ck_rows.append({
#             "Таблица": name,
#             "Ключ": " + ".join(keys),
#             "Дубликатов": None,
#             "Ключ уникален": "Нет нужных столбцов",
#         })
# display(pd.DataFrame(ck_rows).set_index("Таблица"))

# print("\nПРОВЕРКА СВЯЗЕЙ МЕЖДУ ТАБЛИЦАМИ")
# collision_ids = set(dfcol["collision_index"])
# rel_rows = []
# for name, df in [("VEHICLE", dfveh), ("CASUALTY", dfcas)]:
#     child_ids = set(df["collision_index"])
#     orphan_ids = child_ids - collision_ids
#     missing_ids = collision_ids - child_ids
#     rel_rows.append({
#         "Таблица": name,
#         "ДТП без записей": len(missing_ids),
#         "Записи без ДТП": len(orphan_ids),
#         "Всего ДТП": len(collision_ids),
#         "Всего записей": len(child_ids),
#     })
# display(pd.DataFrame(rel_rows).set_index("Таблица"))

# MISSING_RULES = {
#     # ---------- COLLISION ----------
#     "enhanced_collision_severity":                       {"na": [-1]},
#     "local_authority_district":                          {"na": [-1]},
#     "local_authority_ons_district":                      {"na": [-1]},
#     "local_authority_highway":                           {"na": [-1]},
#     "first_road_class":                                  {"na": [-1]},
#     "first_road_number":                                 {"na": [-1]},
#     "road_type":                                         {"na": [-1], "unknown": [9], "normal": [1, 2, 3, 6, 7, 12]},
#     "speed_limit":                                       {"na": [-1], "unknown": [99], "normal": [20, 30, 40, 50, 60, 70]},
#     "junction_detail_historic":                          {"na": [-1], "unknown": [99]},
#     "junction_detail":                                   {"na": [-1], "unknown": [99]},
#     "junction_control":                                  {"na": [-1], "unknown": [9]},
#     "second_road_class":                                 {"na": [-1], "unknown": [9]},
#     "second_road_number":                                {"na": [-1]},
#     "pedestrian_crossing_human_control_historic":        {"na": [-1], "unknown": [9]},
#     "pedestrian_crossing_physical_facilities_historic":  {"na": [-1], "unknown": [9]},
#     "pedestrian_crossing":                               {"na": [-1], "unknown": [99]},
#     "light_conditions":                                  {"na": [-1], "normal": [1, 4, 5, 6, 7]},
#     "weather_conditions":                                {"na": [-1], "unknown": [9], "normal": [1, 2, 3, 4, 5, 6, 7, 8]},
#     "road_surface_conditions":                           {"na": [-1], "unknown": [9], "normal": [1, 2, 3, 4, 5, 6, 7]},
#     "special_conditions_at_site":                        {"na": [-1], "unknown": [9]},
#     "carriageway_hazards_historic":                      {"na": [-1], "unknown": [9]},
#     "carriageway_hazards":                               {"na": [-1], "unknown": [99]},
#     "urban_or_rural_area":                               {"na": [-1], "normal": [1, 2, 3]},
#     "did_police_officer_attend_scene_of_collision":      {"na": [-1]},
#     "did_police_officer_attend_scene_of_accident":       {"na": [-1]},
#     "trunk_road_flag":                                   {"na": [-1]},
#     # ---------- VEHICLE ----------
#     "vehicle_type":                                      {"na": [-1], "unknown": [99]},
#     "towing_and_articulation":                           {"na": [-1], "unknown": [9]},
#     "vehicle_manoeuvre_historic":                        {"na": [-1], "unknown": [99]},
#     "vehicle_manoeuvre":                                 {"na": [-1], "unknown": [99]},
#     "vehicle_direction_from":                            {"na": [-1], "unknown": [9]},
#     "vehicle_direction_to":                              {"na": [-1], "unknown": [9]},
#     "vehicle_location_restricted_lane_historic":         {"na": [-1], "unknown": [99]},
#     "vehicle_location_restricted_lane":                  {"na": [-1], "unknown": [99]},
#     "junction_location":                                 {"na": [-1], "unknown": [9]},
#     "skidding_and_overturning":                          {"na": [-1], "unknown": [9]},
#     "hit_object_in_carriageway":                         {"na": [-1], "unknown": [99]},
#     "vehicle_leaving_carriageway":                       {"na": [-1], "unknown": [9]},
#     "hit_object_off_carriageway":                        {"na": [-1], "unknown": [99]},
#     "first_point_of_impact":                             {"na": [-1], "unknown": [9]},
#     "vehicle_left_hand_drive":                           {"na": [-1], "unknown": [9]},
#     "journey_purpose_of_driver_historic":                {"na": [-1], "unknown": [6]},
#     "journey_purpose_of_driver":                         {"na": [-1], "unknown": [6]},
#     "sex_of_driver":                                     {"na": [-1], "unknown": [3]},
#     "age_of_driver":                                     {"na": [-1]},
#     "age_band_of_driver":                                {"na": [-1]},
#     "engine_capacity_cc":                                {"na": [-1]},
#     "propulsion_code":                                   {"na": [-1]},
#     "generic_make_model":                                {"na": [-1]},
#     "driver_imd_decile":                                 {"na": [-1]},
#     "driver_distance_banding":                           {"na": [-1]},
#     # ---------- CASUALTY ----------
#     "sex_of_casualty":                                   {"na": [-1], "unknown": [9]},
#     "age_of_casualty":                                   {"na": [-1]},
#     "age_band_of_casualty":                              {"na": [-1]},
#     "enhanced_casualty_severity":                        {"na": [-1]},
#     "pedestrian_location":                               {"na": [-1], "unknown": [10]},
#     "pedestrian_movement":                               {"na": [-1], "unknown": [9]},
#     "car_passenger":                                     {"na": [-1], "unknown": [9]},
#     "bus_or_coach_passenger":                            {"na": [-1], "unknown": [9]},
#     "pedestrian_road_maintenance_worker":                {"na": [-1], "unknown": [2]},
#     "casualty_type":                                     {"na": [-1], "unknown": [99]},
#     "casualty_imd_decile":                               {"na": [-1]},
#     "casualty_distance_banding":                         {"na": [-1]},
# }


# def build_missing_report(tables: dict, rules: dict) -> pd.DataFrame:
#     rows = []
#     for table_name, df in tables.items():
#         n = len(df)
#         for col, spec in rules.items():
#             if col not in df.columns:
#                 continue

#             s = df[col]
#             s_num = pd.to_numeric(s, errors="coerce")

#             na_codes = spec.get("na", [])
#             unknown_codes = spec.get("unknown", [])

#             pct_nan = (s.isna().mean() * 100) if n else 0.0
#             pct_na = (s_num.isin(na_codes).mean() * 100) if (n and na_codes) else 0.0
#             pct_unknown = (s_num.isin(unknown_codes).mean() * 100) if (n and unknown_codes) else 0.0

#             rows.append({
#                 "Таблица": table_name,
#                 "Колонка": col,
#                 "% NaN": round(pct_nan, 2),
#                 "% NA (-1)": round(pct_na, 2),
#                 "% unknown (9/99)": round(pct_unknown, 2),
#             })

#     report = pd.DataFrame(rows)
#     if not report.empty:
#         report = report.sort_values(
#             by=["Таблица", "% NA (-1)", "% unknown (9/99)", "% NaN"],
#             ascending=[True, False, False, False],
#         ).reset_index(drop=True)
#     return report


# print("\n============= ПРОПУСКИ: NaN / NA-коды / unknown-коды =================")
# missing_report = build_missing_report(tables, MISSING_RULES)
# display(missing_report)


# def check_col(df: pd.DataFrame, col: str, rules: dict) -> tuple[str, list]:
#     if col not in df.columns:
#         return f"{col}: колонки нет в таблице — НЕ ОК", []
#     spec = rules.get(col)
#     if spec is None:
#         return f"{col}: нет правил в MISSING_RULES — НЕ ОК", []

#     allowed = set()
#     for key in ("na", "unknown", "normal"):
#         allowed |= set(spec.get(key, []))

#     s = pd.to_numeric(df[col], errors="coerce").dropna()
#     unexpected = sorted(set(s.unique()) - allowed)

#     if unexpected:
#         return f"{col}: НЕ ОК", unexpected
#     return f"{col}: всё ок", []


# print("=====Checking ranges and anomalous values=====")

# check_cols = [
#     "speed_limit",
#     "light_conditions",
#     "weather_conditions",
#     "road_type",
#     "road_surface_conditions",
#     "urban_or_rural_area",
# ]

# show_distribution = {
#     "speed_limit",
#     "light_conditions",
# }

# all_ok = True
# for col in check_cols:
#     status, bad = check_col(dfcol, col, MISSING_RULES)
#     print(f"\n{status}")
#     if bad:
#         all_ok = False
#         print(f"    недопустимые значения: {bad}")

#     if col in show_distribution:
#         vc = (
#             dfcol[col]
#             .value_counts(dropna=False)
#             .rename_axis("значение")
#             .reset_index(name="кол-во")
#         )
#         vc["доля, %"] = (vc["кол-во"] / len(dfcol) * 100).round(2)
#         print(f"    распределение {col} ({len(dfcol)} строк):")
#         display(vc)

# print()
# if all_ok:
#     print("ИТОГ: всё норм — все значения во всех проверенных колонках допустимы.")
# else:
#     print("ИТОГ: НЕ норм — есть недопустимые значения (см. выше).")

print('====Diapazons====')
cheks = ['number_of_vehicles', 'number_of_casualties', 'latitude',
        'longitude', 'location_easting_osgr','location_northing_osgr']

display(dfcol[cheks].describe().T)
top10 = dfcol['number_of_casualties'].sort_values(ascending=False).head(10)
print(top10)
for idx, val in top10.items():
    idx = dfcol.loc[idx, 'collision_index']
    actual = (dfcas['collision_index'] == idx).sum()
    status = "настоящее" if actual == val else "ОШИБКА ВВОДА"
    print(f"collision_index={idx}: заявлено={val}, в dfcas={actual} -> {status}")


====Diapazons====


,count,mean,std,min,25%,50%,75%,max
number_of_vehicles,513801.0,1.824179,0.687802,1.000000,1.000000,2.000000,2.000000,2.600000e+01
number_of_casualties,513801.0,1.270572,0.726348,1.000000,1.000000,1.000000,1.000000,1.420000e+02
latitude,513748.0,52.366318,1.312262,49.912210,51.462346,51.843188,53.334331,6.050001e+01
longitude,513748.0,-1.211037,1.355154,-7.486852,-2.111008,-1.106897,-0.133341,1.759829e+00
location_easting_osgr,513748.0,454945.487375,92658.538917,67564.000000,392630.000000,460443.500000,529549.250000,6.553450e+05
location_northing_osgr,513748.0,275427.620133,145647.350208,10211.000000,175147.000000,217372.500000,382395.000000,1.179892e+06


290032    142
275000     70
11401      32
22266      25
271843     22
125955     22
63841      21
32669      21
81327      20
94920      20
Name: number_of_casualties, dtype: int64
collision_index=2025052501204: заявлено=142, в dfcas=142 -> настоящее
collision_index=2023520300610: заявлено=70, в dfcas=70 -> настоящее
collision_index=2025041681268: заявлено=32, в dfcas=32 -> настоящее
collision_index=2025520501079: заявлено=25, в dfcas=25 -> настоящее
collision_index=202163CF00721: заявлено=22, в dfcas=22 -> настоящее
collision_index=2025351616661: заявлено=22, в dfcas=22 -> настоящее
collision_index=2024461446505: заявлено=21, в dfcas=21 -> настоящее
collision_index=2024471416027: заявлено=21, в dfcas=21 -> настоящее
collision_index=2024141499595: заявлено=20, в dfcas=20 -> настоящее
collision_index=2025340NN2152: заявлено=20, в dfcas=20 -> настоящее


размеры трёх таблиц (513 801 / 937 265 / 652 821 строк);
годы 2021-2025, collision_index уникален;
53 ДТП без координат (во всех четырёх колонках сразу);
ID во всех таблицах совпадают, сирот нет.